### Упражнение №1
Создайте класс Vector с полями x, y, z, определите для него конструктор, метод \__abs__ (модуль), необходимые арифметические операции:
- сложение + 
- вычитание -
- \* как скалярное произведение [вектор с вектором]
- \* на число [вектор и число])
Реализуйте конструктор, который принимает строку в формате {x, y, z}. Учтите, что в векторе могут лежать только числа (сделайте assert на то, что x,y,z это числа). 

### Упражнение №1.1
Используя класс Vector выведите координаты центра масс данного множества точек.

### Упражнение №1.2
Среди данных точек найдите три точки, образующие треугольник с наибольшей площадью. Выведите данную площадь.


In [3]:
class Vector:
    def __init__(self, x, y, z):
        assert isinstance(x, (int, float)), "x должен быть числом"
        assert isinstance(y, (int, float)), "y должен быть числом"
        assert isinstance(z, (int, float)), "z должен быть числом"
        self.x = x
        self.y = y
        self.z = z

    def str(self):
        return f"{self.x} {self.y} {self.z}"

    #сложение
    def __add__(self, other):
        return Vector(self.x + other.x,
                      self.y + other.y,
                      self.z + other.z)
    #вычитание
    def __sub__(self, other):
        return Vector(self.x - other.x,
                      self.y - other.y,
                      self.z - other.z)
    #умножение на число или скалярное
    def __mul__(self, other):
        if isinstance(other, Vector):
            return (self.x * other.x + self.y * other.y + self.z * other.z)
        else:
            return Vector(self.x * other,
                      self.y * other,
                      self.z * other)

    #abs
    def __abs__(self):
        return ((self.x * self.x + self.y * self.y + self.z * self.z) ** 0.5)

    
a = Vector(1, 2, 3)
b = Vector(4, 5, 6)
print(abs(a))
print(a * b)
print((a * 2).str())
print((a + b).str())

3.7416573867739413
32
2 4 6
5 7 9


2 задание
Я так понял, надо было еще инвертировать ось, чтобы пушка куда надо стреляла

In [ ]:
from random import randrange as rnd, choice
from tkinter import *
import math
 
import time
root = Tk()
fr = Frame(root)
root.geometry('800x600')
canv = Canvas(root, bg='white')
canv.pack(fill=BOTH, expand=1)


class Ball():
    """ Класс Ball описывает мяч. """

    def __init__(self,x=40,y=450):
        """ Конструктор класса Ball
        Args:
        x - начальное положение мяча по горизонтали
        y - начальное положение мяча по вертикали
        """
        self.x = x
        self.y = y
        self.r = 10
        self.vx = 0
        self.vy = 0
        self.color = choice(['blue','green','red','brown'])
        self.id = canv.create_oval(self.x-self.r, self.y-self.r, self.x+self.r, self.y+self.r, fill=self.color)
        self.live = 30

    def set_coords(self):
        canv.coords(self.id, self.x-self.r, self.y-self.r, self.x+self.r, self.y+self.r)

    def move(self):
        """ Метод move описывает перемещение мяча за один кадр перерисовки. То есть, обновляет значения 
        self.x и self.y с учетом скоростей self.vx и self.vy, силы гравитации, действующей на мяч,
            и стен по краям окна (размер окна 800х600).
        """
        self.x += self.vx
        self.vy += 0.5
        self.y += self.vy

        if self.y + self.r >= 600:
            self.y = 600 - self.r
            self.vy = -0.5 * self.vy
            self.vx = 0.8 * self.vx          # трение об пол
            if abs(self.vy) < 1:             # отскок слишком слабый — мяч остановился
                self.live = 0
                canv.delete(self.id)
                balls.remove(self)
                return

        if self.x + self.r >= 800:
            self.x = 800 - self.r
            self.vx = -0.5 * self.vx
        if self.x - self.r <= 0:
            self.x = self.r
            self.vx = -0.5 * self.vx
        if self.y - self.r <= 0:
            self.y = self.r
            self.vy = -0.5 * self.vy

        self.set_coords()

    def hittest(self,ob):
        """ Функция проверяет сталкивалкивается ли данный обьект с целью, описываемой в обьекте ob.

        Args:
            ob: Обьект, с которым проверяется столкновение.
        Returns:
            Возвращает True в случае столкновения мяча и цели. В противном случае возвращает False.
        """
        dx = self.x - ob.x
        dy = self.y - ob.y
        return dx**2 + dy**2 <= (self.r + ob.r)**2

class gun():
    """ Класс gun описывает пушку. """
    def __init__(self,x=40,y=450):
        self.f2_power = 10
        self.f2_on = 0
        self.an = 1
        self.id = canv.create_line(20,450,50,420,width=7) # FIXME: don't know how to set it...
         
    def fire2_start(self,event):
        self.f2_on = 1
 
    def fire2_end(self,event):
        """ Выстрел мячом происходит при отпускании кнопки мыши.
        Начальные значения компонент скорости мяча vx и vy зависят от положения мыши.
        """
        global balls, bullet
        bullet += 1
        new_ball = Ball()
        new_ball.r += 5
        self.an = math.atan((event.y-new_ball.y)/(event.x-new_ball.x))
        new_ball.vx = self.f2_power*math.cos(self.an)
        new_ball.vy = self.f2_power*math.sin(self.an)
        balls += [new_ball]
        self.f2_on = 0
        self.f2_power = 10
 
 
    def targetting (self,event=0):
        """ Прицеливание. Зависит от положения мыши.
        """
        if event:
            self.an = math.atan((event.y-450)/(event.x-20))    
        if self.f2_on:
            canv.itemconfig(self.id,fill = 'orange')
        else:
            canv.itemconfig(self.id,fill = 'black')
        canv.coords(self.id, 20, 450, 20 + max(self.f2_power, 20) * math.cos(self.an), 450 + max(self.f2_power, 20) * math.sin(self.an))
         

    def power_up(self):
        if self.f2_on:
            if self.f2_power < 100:
                self.f2_power += 1
            canv.itemconfig(self.id,fill = 'orange')
        else:
            canv.itemconfig(self.id,fill = 'black')
        
class target():
    """ Класс target описывает цель. """ 
    def __init__(self,x=40,y=450):
        self.points = 0
        self.live = 1
        self.id = canv.create_oval(0,0,0,0)
        self.id_points = canv.create_text(30,30,text = self.points,font = '28')
        self.new_target()
         
    def new_target(self):
        """ Инициализация новой цели. """
        x = self.x = rnd(600,780)
        y = self.y = rnd(300,550)
        r = self.r = rnd(2,50)
        color = self.color = 'red'
        canv.coords(self.id, x-r,y-r,x+r,y+r)
        canv.itemconfig(self.id, fill = color)
         
    def hit(self,points = 1):
        """ Попадание шарика в цель. """
        canv.coords(self.id,-10,-10,-10,-10)
        self.points += points
        canv.itemconfig(self.id_points, text = self.points)


t1 = target()
screen1 = canv.create_text(400,300, text = '',font = '28')
g1 = gun()
bullet = 0
balls = []



def new_game(event=''):
    global gun, t1, screen1, balls, bullet
    t1.new_target()
    bullet = 0
    balls = []
    canv.bind('<Button-1>', g1.fire2_start)
    canv.bind('<ButtonRelease-1>', g1.fire2_end)
    canv.bind('<Motion>', g1.targetting)
 
    z = 0.03
    t1.live = 1
    while t1.live or balls:
        for b in balls[:]:
            b.move()
            if b.hittest(t1) and t1.live:
                t1.live = 0
                t1.hit()
                canv.bind('<Button-1>', '')
                canv.bind('<ButtonRelease-1>', '')
                canv.itemconfig(screen1, text = 'Вы уничтожили цель за ' + str(bullet) + ' выстрелов')
        canv.update()
        time.sleep(0.03)
        g1.targetting()
        g1.power_up()
    canv.itemconfig(screen1, text = '')
    canv.delete(gun)
    root.after(750,new_game)

new_game()   
 
tk.mainloop()

NameError: name 'tk' is not defined

: 

1. Проверьте, представляет ли массив введённый с клавиатуры список минимальную кучу или нет. 
Если да, то выведите 1, если нет - 0.

In [ ]:
def is_min_heap(s):
    arr = list(map(int, s.split()))
    n = len(arr)

    for i in range(n // 2):
        left = 2 * i + 1
        right = 2 * i + 2

        if left < n and arr[left] < arr[i]:
            return False
        if right < n and arr[right] < arr[i]:
            return False

    return True


2. В этой задаче вам даётся три списка студентов:
- владеющих французским языком
- умеющих плавать
- владеющих игрой на фортепиано
Вам необходимо определить список пловцов-пианистов, не владеющих французским.

Формат входных данных:

Три строки. В каждой через пробел записаны номера зачёток студентов. Первая строка посвящена владеющим французским, вторая пловцам и третья пианистам.

Формат выходных данных: 

Одна строка c номерами зачёток через пробел по возрастанию.

Примеры:

Ввод данных:

1 2 5 7 8 9

3 4 8 2 10

10 3 2 8 5

Вывод данных:

3 10

In [ ]:
f = input()
s = input()
p = input()


def non_french(f, s, p):
    F = set(map(int, f.split()))
    S = set(map(int, s.split()))
    P = set(map(int, p.split()))

    result = (S & P) - F

    return sorted(result)


ans = non_french(f, s, p)
print(ans)

[3, 10]


3. Популярная газета объявила конкурс на выбор лучшего фильма, для которого стоит снять продолжение. На выбор читателей было предложено 10 фильмов. Вам предлагается написать эффективную, в том числе и по используемой памяти, программу, которая будет статистически обрабатывать результаты sms-голосования по этому вопросу, чтобы определить популярность того или иного фильма. Следует учитывать, что количество голосов в списке может быть очень велико.

Программа должна вывести список всех фильмов, встречающихся в списке, в порядке убывания (невозрастания) количества отданных за них голосов с указанием этого количества голосов. Название каждого фильма должно быть выведено только один раз.

Формат входных данных: 

На вход программе в первой строчке подается количество пришедших sms-сообщений N. В каждой из последующих N строк записано название фильма. При этом название фильма может содержать пробелы.

Формат выходных данных:

Название фильма и число голосов, отданых за фильм, разделяются пробелом.

Примеры:

Ввод:

6

Белое солнце пустыни

Бриллиантовая рука

Белое солнце пустыни

Белое солнце пустыни

Гараж

Бриллиантовая рука

Вывод:

Белое солнце пустыни 3

Бриллиантовая рука 2

Гараж 1 

In [4]:
N = int(input())
arr = []
for i in range(N):
    x = input().strip()
    arr.append(x)


def film_stat():
    all_films = {}
    for i in range(N):
        if arr[i] not in all_films:
            all_films[arr[i]] = 1
        else:
            all_films[arr[i]] += 1

    sorted_films = sorted(all_films.items(), key=lambda x: x[1], reverse=True)  # сортировка, насколько я знаю по умолчанию за NlogN работает

    for name, count in sorted_films:
        print(name, count)


film_stat()
# итого должно работать за N + KlogK

в 3


5. Реализуйте алгоритм сортировки кучей. На вход подаётся неотсортированный список, на выходе должен получиться отсортированные список.

Формат входных данных:

6 4 7 1 9 -2

Формат выходных данных:

-2 1 4 6 7 9

In [10]:
# родитель меняется с большим дочерним элементом
def swap(arr, i, j):
    arr[i], arr[j] = arr[j], arr[i]


# гарантирует что дети больше родителей
def siftDown(arr, i, upper):
    while (True):  # цикл работает пока мы его не завершим
        l, r = 2 * i + 1, 2 * i + 2
        if max(l, r) < upper:  # если дети у верхнего узна вообще существуют
            if arr[i] >= max(arr[l], arr[r]):
                break
            elif arr[l] > arr[r]:
                swap(arr, i, l)
                i = l  # спускаемся ниже, тк для верхнего родителя все выполнено
            else:
                swap(arr, i, r)
                i = r  # спускаемся но уже к правому элементу
        elif l < upper:
            if arr[l] > arr[i]:
                swap(arr, i, l)
                i = l
            else:
                break
        elif r < upper:
            if arr[r] > arr[i]:
                swap(arr, i, r)
                i = r
            else:
                break
        else:
            break


# короче проходим по каждому родителю снизу вверх и дл каждого родителя
# вызываем siftdown, таким образом для самых нижних уровней окажется что
# они являются идеальной подкучей тривиально, потом когда проходим по родителям на
# более высоких уровнях так же вызывается siftdown, однако число, на которое заменили
# родителя из самой нижней кучи может стать меньш одного из его детей, соответственно
# поэтому siftdown и продолжает работать, чтобы в итоге массив стал кучей.
# потом меняются местами последний элемент массива и корень (самый большой элемент)
# этот самый большой элемент сохраняется и убирается из дальнейшего рассмотрения

def heapsort(arr):
    for j in range((len(arr) - 2) // 2, -1, -1):
        siftDown(arr, j, len(arr))

    for end in range(len(arr) - 1, 0, -1):
        swap(arr, 0, end)
        siftDown(arr, 0, end)


a = [2, 3, -6, 7, 3, -100, 300]
heapsort(a)
print(a)

[-100, -6, 2, 3, 3, 7, 300]
